# Structural Connectivity from Diffusion MRI

**Author**: Joan Amos, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/Joanone"><img src="https://img.shields.io/badge/-Joan_Amos-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 16/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

## Purpose

**Structural connectivity** maps the white matter pathways connecting different brain regions using diffusion MRI (dMRI) tractography. The resulting **connectome** - a matrix of connection strengths between brain regions - is a fundamental tool in network neuroscience for studying brain organisation, development, and disease.

This tutorial walks through a complete structural connectivity pipeline using a single subject from the Human Connectome Project (HCP), covering:

1. Pre-processing diffusion data with MRtrix3, FSL, and AFNI
2. Constructing a whole-brain tractogram
3. Refining streamlines with SIFT2
4. Building a structural connectome using the Desikan-Killiany atlas

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Perform fibre orientation distribution estimation from dMRI data
- Generate a whole-brain tractogram using probabilistic tractography
- Construct and visualise a structural connectome matrix
:::


## Citation and Resources

### Tools used in this workflow

__MRtrix3__
: Tournier, J.-D., et al. (2019). MRtrix3: A fast, flexible and open software framework for medical image processing and visualisation. *NeuroImage*, 202, 116137. [https://doi.org/10.1016/j.neuroimage.2019.116137](https://doi.org/10.1016/j.neuroimage.2019.116137)

__FSL__
: Jenkinson, M., et al. (2012). FSL. *NeuroImage*, 62(2), 782–790. [https://doi.org/10.1016/j.neuroimage.2011.09.015](https://doi.org/10.1016/j.neuroimage.2011.09.015)

### Dataset

__Human Connectome Project__
: Data available from [db.humanconnectome.org](https://db.humanconnectome.org)

### Educational resources

- [HCP dMRI connectome pipeline](https://github.com/civier/HCP-dMRI-connectome)
- [Andy's Brain Book - MRtrix tutorial](https://andysbrainbook.readthedocs.io/en/latest/MRtrix/MRtrix_Course/MRtrix_00_Diffusion_Overview.html)
- [MRtrix3 connectome documentation](https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/structural_connectome.html)


## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment
- [ ] Sufficient disk space (the pipeline generates several GB of intermediate files)
- [ ] Familiarity with basic terminal commands and dMRI concepts


## Download demo data

This tutorial uses minimally preprocessed T1w and diffusion data from a single HCP subject (100307). Open a terminal in Neurodesktop and run the following commands to download and prepare the data:

```bash
pip install gdown
mkdir -p ~/neurodesktop-storage/Test/100307

gdown 1bj-Tz2xLGn0L05CveOYnzTgkewc_IU5F -O ~/neurodesktop-storage/Test/T1w_data.zip
gdown 193ZaUmXbT59IoXoAJ-Y1pf_wgdSHdKpR -O ~/neurodesktop-storage/Test/diffusion_data.zip

unzip -o ~/neurodesktop-storage/Test/T1w_data.zip -d ~/neurodesktop-storage/Test/
unzip -o ~/neurodesktop-storage/Test/diffusion_data.zip -d ~/neurodesktop-storage/Test/
```

The zip files extract into subdirectories under `100307/`. Copy the required files into the working directory so the pipeline commands can find them:

```bash
cp ~/neurodesktop-storage/Test/100307/T1w/aparc+aseg.nii.gz \
   ~/neurodesktop-storage/Test/100307/T1w/T1w_acpc_dc_restore_brain.nii.gz \
   ~/neurodesktop-storage/Test/100307/

cp ~/neurodesktop-storage/Test/100307/T1w/Diffusion/data.nii.gz \
   ~/neurodesktop-storage/Test/100307/T1w/Diffusion/bvals \
   ~/neurodesktop-storage/Test/100307/T1w/Diffusion/bvecs \
   ~/neurodesktop-storage/Test/100307/
```

Confirm the files are in place:

```bash
ls ~/neurodesktop-storage/Test/100307/
```

You should see `aparc+aseg.nii.gz`, `T1w_acpc_dc_restore_brain.nii.gz`, `data.nii.gz`, `bvals`, and `bvecs` alongside the extracted directories.

![Input files confirmed in terminal](/static/tutorials/structural_imaging/structuralconnectivity/01_start.png)
*Confirming the input files are present.*

## Activate software tools

Load MRtrix3, FSL, and AFNI from the Neurodesk terminal. For reproducibility, this tutorial uses specific versions:

```bash
ml mrtrix3/3.0.3
ml fsl/6.0.5.1
ml afni/21.2.00
```

![Activating software in Neurodesk](/static/tutorials/structural_imaging/structuralconnectivity/02_activate_softwares.png)
*Loading the required software modules.*


## Step 1: Pre-processing

Navigate to the subject directory:

```bash
cd ~/neurodesktop-storage/Test/100307
```

### Extract diffusion data

Extract `data.nii.gz` to enable memory-mapping (the extracted files are ~4.5 GB):

```bash
mrconvert data.nii.gz data.mif -fslgrad bvecs bvals
```

![Pre-processing: mrconvert](/static/tutorials/structural_imaging/structuralconnectivity/03_preproc.png)
*Converting the diffusion data to MRtrix format.*


### Extract response functions

Estimate the response functions for white matter, grey matter, and CSF:

```bash
dwi2response dhollander data.mif wm.txt gm.txt csf.txt
```

![Response function estimation](/static/tutorials/structural_imaging/structuralconnectivity/05_preproc.png)
*Estimating tissue response functions.*


### Generate brain mask and FODs

Create a brain mask from the diffusion data:

```bash
dwi2mask data.mif mask.mif
```

Generate fibre orientation distributions (FODs):

```bash
dwi2fod msmt_csd data.mif \
    wm.txt wmfod.mif \
    gm.txt gmfod.mif \
    csf.txt csffod.mif \
    -mask mask.mif
```

![FOD generation](/static/tutorials/structural_imaging/structuralconnectivity/08_preproc.png)
*Generating fibre orientation distributions.*


### Normalise FODs and generate tissue maps

Perform intensity normalisation:

```bash
mtnormalise wmfod.mif wmfod_norm.mif \
    gmfod.mif gmfod_norm.mif \
    csffod.mif csffod_norm.mif \
    -mask mask.mif
```

Generate a five-tissue-type segmentation:

```bash
5ttgen fsl T1w_acpc_dc_restore_brain.nii.gz 5tt.mif -premasked
```

![5TT generation](/static/tutorials/structural_imaging/structuralconnectivity/10_preproc.png)
*Generating the 5-tissue-type image.*


### Coregister anatomical to diffusion space

Extract the mean B0 volume and the grey matter segmentation, then coregister:

```bash
dwiextract data.mif - -bzero | mrmath - mean mean_b0.mif -axis 3
mrconvert mean_b0.mif mean_b0.nii.gz
mrconvert 5tt.mif 5tt.nii.gz

fslroi 5tt.nii.gz 5tt_GM.nii.gz 0 1

flirt -in 5tt_GM.nii.gz -ref mean_b0.nii.gz -omat T1_to_DWI.mat -dof 6

transformconvert T1_to_DWI.mat 5tt_GM.nii.gz mean_b0.nii.gz \
    flirt_import T1_to_DWI_mrtrix.txt

mrtransform 5tt.mif 5tt_coreg.mif \
    -linear T1_to_DWI_mrtrix.txt -inverse
```

![Coregistration](/static/tutorials/structural_imaging/structuralconnectivity/15_preproc.png)
*Coregistering the anatomical image to diffusion space.*


### Create the grey/white matter interface seed

```bash
5tt2gmwmi 5tt_coreg.mif gmwmi_seed.mif
```

![GM/WM interface](/static/tutorials/structural_imaging/structuralconnectivity/16_preproc.png)
*Creating the grey matter / white matter interface seed boundary.*


## Step 2: Tractogram construction

Generate a whole-brain tractogram using probabilistic tractography (iFOD2 algorithm). Here we use 10 million streamlines to save computational time:

```bash
tckgen -act 5tt_coreg.mif \
    -backtrack \
    -seed_gmwmi gmwmi_seed.mif \
    -select 10000000 \
    wmfod_norm.mif tracks_10M.tck
```

:::{note}
This step may take a while depending on your hardware. Wait for it to reach 100% before proceeding.
:::

![Tractogram generation](/static/tutorials/structural_imaging/structuralconnectivity/17_tractogram.png)
*Generating the whole-brain tractogram.*


## Step 3: SIFT2 streamline weighting

Refine the tractogram with SIFT2 to counterbalance reconstruction biases. This creates a text file containing a weight for each streamline:

```bash
tcksift2 -act 5tt_coreg.mif \
    tracks_10M.tck wmfod_norm.mif \
    sift2_weights.txt
```

![SIFT2 weighting](/static/tutorials/structural_imaging/structuralconnectivity/18_sift2.png)
*Running SIFT2 to compute streamline weights.*


## Step 4: Connectome construction

We use the Desikan-Killiany atlas (84 regions, cortical and subcortical) to define brain regions.

### Prepare atlas files

Copy the required lookup table and mapping files from the FreeSurfer and MRtrix3 containers:

```bash
cp /opt/freesurfer-7.2.0/FreeSurferColorLUT.txt ~/neurodesktop-storage/Test/100307/
cp /opt/mrtrix3-3.0.3/share/mrtrix3/labelconvert/fs_default.txt ~/neurodesktop-storage/Test/100307/
```

:::{note}
The exact paths may differ depending on which software versions you loaded. Adjust the version numbers in the paths above to match.
:::

![Copying atlas files](/static/tutorials/structural_imaging/structuralconnectivity/20_connectome.png)
*Copying the lookup table and mapping files.*


### Convert parcellation and build the connectome

Convert the FreeSurfer parcellation to MRtrix3 format:

```bash
labelconvert aparc+aseg.nii.gz \
    FreeSurferColorLUT.txt fs_default.txt \
    nodes.mif -force
```

Coregister the parcellation to diffusion space:

```bash
mrtransform nodes.mif nodes_coreg.mif \
    -linear T1_to_DWI_mrtrix.txt \
    -inverse -datatype uint32
```

Build the structural connectome:

```bash
tck2connectome -symmetric -zero_diagonal \
    -scale_invnodevol \
    -tck_weights_in sift2_weights.txt \
    tracks_10M.tck nodes_coreg.mif nodes.csv -force
```

![Building the connectome](/static/tutorials/structural_imaging/structuralconnectivity/24_connectome.png)
*Building the structural connectome matrix.*


## Viewing the connectome

The resulting `nodes.csv` file is a symmetric matrix that can be visualised in MATLAB, Python, or any matrix viewer.

For example, in MATLAB:

```matlab
connectome = importdata('nodes.csv');
imagesc(connectome, [0 1])
colorbar
```

Or in Python:

```python
import numpy as np
import matplotlib.pyplot as plt

connectome = np.loadtxt('nodes.csv', delimiter=',')
plt.imshow(connectome, vmin=0, vmax=1, cmap='hot')
plt.colorbar()
plt.title('Structural Connectome (Desikan-Killiany)')
plt.show()
```

![Structural connectome matrix](/static/tutorials/structural_imaging/structuralconnectivity/25_connectome.png)
*The structural connectome matrix for a single HCP subject.*


## Summary

In this tutorial you:

1. Pre-processed diffusion MRI data (response estimation, FOD generation, coregistration)
2. Constructed a whole-brain tractogram with 10 million streamlines
3. Applied SIFT2 to compute biologically meaningful streamline weights
4. Built a structural connectome using the Desikan-Killiany atlas

:::{seealso}
- [FreeSurfer tutorial](freesurfer.ipynb) - for cortical reconstruction used in parcellation
:::
